<a href="https://colab.research.google.com/github/Kunal0110/mini_projects_ML/blob/mini_feature/deep-learning/NLP/TrainerAPI/SQuAD_Fine_Tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer, DefaultDataCollator
import numpy as np

# 1. Load Data
print("Loading SQuAD dataset...")
squad = load_dataset("squad", split="train[:5000]") # Subset for speed
squad = squad.train_test_split(test_size=0.2)       # Split into train/val

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

Loading SQuAD dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def preprocess_function(examples):
  questions = [q.strip() for q in examples["question"]]
  inputs = tokenizer(
      questions,
      examples["context"],
      max_length=384,
      truncation="only_second",
      padding="max_length",
      return_overflowing_tokens=True,
      return_offsets_mapping=True,
  )

  offset_mapping = inputs.pop("offset_mapping")
  sample_map = inputs.pop("overflow_to_sample_mapping")
  answers = examples["answers"]
  start_positions = []
  end_positions = []

  for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]

        # Start/End char index of the answer in the text
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])

        # Locate the context in the tokens (0=question, 1=context usually)
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context within the token list
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If answer is not fully inside the context chunk, label it (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Map char positions to token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

  inputs["start_positions"] = start_positions
  inputs["end_positions"] = end_positions
  return inputs

print("Tokenizing data (this handles strides and mapping)...")
tokenized_squad = squad.map(preprocess_function, batched=True, remove_columns=squad["train"].column_names)

Tokenizing data (this handles strides and mapping)...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir = "./results_squad",
    eval_strategy = "epoch",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    weight_decay = 0.01,
    save_strategy = "no",
    report_to = "none",
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_squad["train"],
    eval_dataset = tokenized_squad["test"],
    processing_class = tokenizer,
    data_collator = DefaultDataCollator(),
)

In [ ]:
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss
1,No log,2.185280
2,2.638300,1.672414
3,2.638300,1.619116


TrainOutput(global_step=765, training_loss=2.1893508412479576, metrics={'train_runtime': 485.9745, 'train_samples_per_second': 25.156, 'train_steps_per_second': 1.574, 'total_flos': 1197925610918400.0, 'train_loss': 2.1893508412479576, 'epoch': 3.0})

In [ ]:
print("Saving model for analysis")
model.save_pretrained("./results_squad/model")
tokenizer.save_pretrained("./results_squad/model")

Saving model for analysis


('./results_squad/model/tokenizer_config.json',
 './results_squad/model/special_tokens_map.json',
 './results_squad/model/vocab.txt',
 './results_squad/model/added_tokens.json',
 './results_squad/model/tokenizer.json')

In [ ]:
model_path = "./results_squad/model"
model = AutoModelForQuestionAnswering.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model.to("cpu")
model.eval()

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

In [ ]:
val_dataset = load_dataset("squad", split="train[:300]")

In [ ]:
def predict_answer(question, context):
    inputs = tokenizer(question, context, return_tensors="pt", max_length=384, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the highest scoring start and end tokens
    answer_start_index = torch.argmax(outputs.start_logits)
    answer_end_index = torch.argmax(outputs.end_logits) + 1

    # Convert tokens back to string
    predict_answer_tokens = inputs.input_ids[0, answer_start_index:answer_end_index]
    return tokenizer.decode(predict_answer_tokens, skip_special_tokens=True)

In [ ]:
print("\n--- ERROR ANALYSIS: Long-Range vs Missing Context ---\n")
print(f"{'ID':<4} | {'Prediction':<20} | {'True Answer':<20} | {'Question'}")
print("-" * 80)

errors_found = 0
target_errors = 20

for i, example in enumerate(val_dataset):
    if errors_found >= target_errors:
        break

    question = example["question"]
    context = example["context"]
    true_answer = example["answers"]["text"][0]

    pred_answer = predict_answer(question, context)

    # Basic normalization to compare (ignore case/punctuation diffs)
    if pred_answer.strip().lower() != true_answer.strip().lower():
        errors_found += 1

        print(f"\n[Error #{errors_found}]")
        print(f"Question:  {question}")
        print(f"Predicted: {pred_answer}")
        print(f"Actual:    {true_answer}")
        print(f"Context Snippet: ...{context[0:200]}...") # Print start of context

        # Analysis Prompt for you
        print(">>> ANALYSIS: Why did it fail?")
        print("    [A] Long-Range Dependency? (Answer is far from the question keywords?)")
        print("    [B] Missing Context? (Was the answer cut off or confusingly phrased?)")


--- ERROR ANALYSIS: Long-Range vs Missing Context ---

ID   | Prediction           | True Answer          | Question
--------------------------------------------------------------------------------

[Error #1]
Question:  What is in front of the Notre Dame Main Building?
Predicted: 
Actual:    a copper statue of Christ
Context Snippet: ...Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta...
>>> ANALYSIS: Why did it fail?
    [A] Long-Range Dependency? (Answer is far from the question keywords?)
    [B] Missing Context? (Was the answer cut off or confusingly phrased?)

[Error #2]
Question:  The Basilica of the Sacred heart at Notre Dame is beside to which structure?
Predicted: basilica of the sacred heart. immediately behind the basilica is the grotto
Actual:    the Main Building
Context Snippet: ...Architecturally, the school has a Catho